# 📓 Semana 1 · Dia 2 — Arquitetura Lakehouse, Unity Catalog e Volumes

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (arquitetura + UC) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Diagrama mental da arquitetura + notas no caderno |

---


## 📖 Teoria — Por que Lakehouse?

**Data Warehouse** (anos 90): excelente para SQL e BI, mas caro, proprietário e péssimo para IA/ML.
**Data Lake** (anos 2010): barato, escala, lê qualquer formato — mas vira um *pântano de dados* (data swamp): sem transações, sem consistência, sem governança.

**Lakehouse** (Databricks, 2020) = a união dos dois: dados abertos (Parquet/Delta) em armazenamento barato, com **transações ACID**, versionamento e governança de warehouse, e motores para SQL, Python, R e IA no mesmo dado.

**As 7 características do Lakehouse**:
1. Transações ACID (Atomicidade, Consistência, Isolamento, Durabilidade)
2. Schema enforcement e evolução
3. Dados abertos e acessíveis (Parquet/Delta, sem lock-in)
4. Suporte a BI direto na fonte
5. Storage separado do compute (S3/ADLS/GCS + clusters)
6. Versionamento de dados (Time Travel)
7. Governança unificada (Unity Catalog)


## 📖 Teoria — Unity Catalog — governança em 3 níveis

O **Unity Catalog (UC)** é a camada de governança que organiza TUDO o que existe no workspace em um **namespace de 3 níveis**:

```
catálogo.schema.objeto

workspace.bronze.vendas_bronze
```

| Nível | Exemplo | O que guarda |
|---|---|---|
| Catálogo | `workspace` | Agrupamento máximo (metastore); `workspace` é o padrão da Free Edition |
| Schema | `bronze`, `prata`, `ouro` | Agrupamento lógico de objetos relacionados |
| Objeto | `vendas_bronze` | Tabela, view, volume, função, modelo, ... |

> 🎯 **Dica de prova (DEA)**: em 2026 o Unity Catalog vale **~30% da prova**. A nomenclatura de 3 níveis (`catalog.schema.table`), permissões (GRANT/REVOKE), external locations e dynamic views são os tópicos mais cobrados. DBFS é legado — o recomendado é **Volumes**.


## 📖 Teoria — DBFS vs Volumes

**DBFS (Databricks File System)**: sistema de arquivos montado no cluster. Fácil para começar, mas é **legado** — difícil de governar e não escala para equipes.

**Volumes (Unity Catalog)**: diretórios governados dentro do UC, com permissões, linhagem e controle de acesso. É o **padrão 2026** para armazenar arquivos (deltas, parquets, modelos, dados brutos).

```
/Volumes/catalogo/schema/volume/caminho/arquivo.csv
```


### 💻 Na prática — Criando schemas do projeto (Medallion)

Vamos criar a estrutura de pastas lógica do projeto — os 3 schemas da arquitetura Medallion (Bronze → Prata → Ouro). Isso prepara o terreno para os próximos dias.


In [ ]:
%sql
-- Criação dos schemas da arquitetura Medallion
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.prata;
CREATE SCHEMA IF NOT EXISTS workspace.ouro;
SHOW SCHEMAS IN workspace

### 💻 Na prática — Volumes na prática

Crie um volume gerenciado para guardar arquivos brutos do curso (dados do projeto).


In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.vol_dados_curso;
SHOW VOLUMES IN workspace.bronze

In [ ]:
# O caminho de um volume montado é acessível do Python assim:
print("/Volumes/workspace/bronze/vol_dados_curso")
display(spark.sql("SHOW VOLUMES IN workspace.bronze"))

> 🎯 **Dica de prova**: A prova DEA distingue **managed tables** (gerenciadas pelo UC, criadas com `CREATE TABLE`) de **external tables** (apontam para storage fora do UC, exigem external location — que não existe na Free Edition). Memorize essa diferença.


## 🎯 Exercícios de fixação

**1.** Explique em 3 frases por que o Lakehouse supera warehouse e data lake.

**2.** Escreva o namespace de 3 níveis da tabela que criaremos: vendas no schema bronze do catálogo workspace.

**3.** Qual a diferença entre managed e external table?

**4.** Por que Volumes são o padrão 2026 e não DBFS?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Lakehouse

Une a abertura e o custo baixo do data lake com ACID, versionamento e governança do warehouse, num único motor para SQL, Python e IA.

**2.** Namespace

`workspace.bronze.vendas_bronze` (catálogo.schema.objeto).

**3.** Managed vs external

Managed: o UC gerencia o armazenamento e o ciclo de vida (DROP apaga dados). External: aponta para storage próprio (S3/ADLS/GCS) via external location; DROP não apaga os arquivos. Na Free Edition só temos managed.

**4.** Volumes vs DBFS

Volumes são governados pelo UC (permissões, linhagem, auditoria) e acessíveis em qualquer compute; DBFS é legado, sem governança fina e com acesso dependente do cluster.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*